In [1]:
import gc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, kendalltau

import os, gc, glob, json
from pathlib import Path

# Load CSV data file

In [2]:
final_df = pd.read_csv("data/disruption_analysis.csv")

In [3]:
final_df = final_df[final_df.doctype == "article"]

final_df = final_df[final_df["team_size"] <= 40]
final_df = final_df[final_df["year"] >= 1971]

final_df = final_df[
    ~((final_df["first_time_author_ratio"] == 1) & (final_df["avg_career_age"] > 0))
]
final_df = final_df[
    ~(
        (final_df["first_time_author_ratio"] == 1)
        & (final_df["senior_author_avg_disruption"] > 0)
    )
]
final_df = final_df[
    ~(
        (final_df["first_time_author_ratio"] == 1)
        & (final_df["mid_career_author_ratio"] > 0)
    )
]
final_df = final_df[
    ~(
        (final_df["first_time_author_ratio"] == 1)
        & (final_df["early_author_avg_disruption"] > 0)
    )
]

final_df["year"] = final_df["year"].astype(int)

final_df["decade_start"] = ((final_df["year"] - 1) // 10) * 10 + 1
final_df["decade_end"] = final_df["decade_start"] + 9
final_df["decade"] = (
    final_df["decade_start"].astype(str) + "-" + final_df["decade_end"].astype(str)
)

# We want to merge mid_career_author_ratio with early_career_author_ratio and delete mid_career_author_ratio
final_df["early_career_author_ratio"] = (
    final_df["early_career_author_ratio"] + final_df["mid_career_author_ratio"]
)

# Handle null values in disruption calculations
final_df["early_author_avg_disruption"] = np.where(
    final_df["early_author_avg_disruption"].isna()
    | final_df["mid_author_avg_disruption"].isna(),
    np.nan,
    (final_df["early_author_avg_disruption"] + final_df["mid_author_avg_disruption"])
    / 2,
)

final_df.drop(
    columns=[
        "decade_start",
        "decade_end",
        "mid_career_author_ratio",
        "mid_author_avg_disruption",
    ],
    inplace=True,
)

final_df["disruption_percentile"] = final_df["disruption"].rank(pct=True) * 100

final_df["citation_count_percentile"] = final_df["citation_count"].rank(pct=True) * 100
final_df["C10_percentile"] = final_df["C10"].rank(pct=True) * 100


final_df["avg_disruption_percentile"] = final_df["avg_disruption"].rank(pct=True) * 100
final_df["avg_citation_count_percentile"] = (
    final_df["avg_citation_count"].rank(pct=True) * 100
)

final_df["Atyp_Median_Z_percentile"] = final_df["Atyp_Median_Z"].rank(pct=True) * 100
final_df["avg_reference_age_percentile"] = (
    final_df["avg_reference_age"].rank(pct=True) * 100
)
final_df["median_reference_age_percentile"] = (
    final_df["median_reference_age"].rank(pct=True) * 100
)
final_df["avg_reference_popularity_percentile"] = (
    final_df["avg_reference_popularity"].rank(pct=True) * 100
)
final_df["median_reference_popularity_percentile"] = (
    final_df["median_reference_popularity"].rank(pct=True) * 100
)

gc.collect()

0

In [4]:
def avg_disruption_group(x):
    if pd.isna(x):  # Handle null values
        return None
    if x < 60:
        return "0-60 percentile"
    elif x < 70:
        return "60-70 percentile"
    elif x < 80:
        return "70-80 percentile"
    elif x < 90:
        return "80-90 percentile"
    else:
        return "90-100 percentile"


final_df["co_authors_disruption_group"] = final_df["avg_disruption_percentile"].apply(
    avg_disruption_group
)
final_df["co_authors_citation_group"] = final_df["avg_citation_count_percentile"].apply(
    avg_disruption_group
)

# Get the avg_disruption values sorted to use as reference scale
avg_disruption_sorted = final_df["avg_disruption"].sort_values().values


# Function to find percentile of a value in the reference distribution
def find_percentile_in_reference(value, reference_array):
    if pd.isna(value):
        return np.nan
    # Find where this value would rank in the reference array
    percentile = (
        np.searchsorted(reference_array, value, side="right")
        / len(reference_array)
        * 100
    )
    return percentile


# Apply to Senior and Early Career Author Disruption - keeping null if original values are null
final_df["senior_author_disruption_percentile"] = np.where(
    final_df["senior_author_avg_disruption"].isna(),
    np.nan,
    final_df["senior_author_avg_disruption"].apply(
        lambda x: find_percentile_in_reference(x, avg_disruption_sorted)
    ),
)

final_df["early_career_disruption_percentile"] = np.where(
    final_df["early_author_avg_disruption"].isna(),
    np.nan,
    final_df["early_author_avg_disruption"].apply(
        lambda x: find_percentile_in_reference(x, avg_disruption_sorted)
    ),
)

# Now apply the bucketing function - this will handle nulls properly
final_df["senior_author_disruption_bucket"] = final_df[
    "senior_author_disruption_percentile"
].apply(avg_disruption_group)
final_df["early_career_disruption_bucket"] = final_df[
    "early_career_disruption_percentile"
].apply(avg_disruption_group)

gc.collect()

0

In [5]:
print(final_df.shape)
gc.collect()

(28042889, 63)


0

In [6]:
def setup_plotting_style():
    plt.figure(figsize=(8, 6), dpi=300)

    sns.set_style("white")

    plt.rcParams["font.size"] = 12
    plt.rcParams["axes.labelsize"] = 14
    plt.rcParams["axes.titlesize"] = 16
    plt.rcParams["xtick.labelsize"] = 12
    plt.rcParams["ytick.labelsize"] = 12
    plt.rcParams["legend.fontsize"] = 12
    plt.rcParams["figure.titlesize"] = 18

    plt.rcParams["axes.linewidth"] = 1.5
    plt.rcParams["grid.linewidth"] = 0.8
    plt.rcParams["lines.linewidth"] = 2.0

    sns.set_palette("colorblind")

    # Save format settings
    plt.rcParams["savefig.format"] = "pdf"
    plt.rcParams["savefig.bbox"] = "tight"
    plt.rcParams["savefig.pad_inches"] = 0.1


setup_plotting_style()

<Figure size 2400x1800 with 0 Axes>

In [7]:
!mkdir Figures

mkdir: Figures: File exists


# Highly disruptive papers by beginner-heavy teams are highly cited